# Geometry 02b — Saturation et Nullstellensatz : l'opération derrière l'astuce

**Public : Licence** — accretion du notebook [02](Geometry-02-From-Equation-To-Proof.ipynb) (*From Equation to Proof*), qu'elle suppose lu ; la series se lit sans ouvrir cette lettre (cf [README](README.md)).


## Ou en sommes-nous ?

Le notebook 02 s'est arrete sur un geste qui a l'air d'un tour de passe-passe : pour sauver le theoreme « *B orthogonal a C et a D implique C, D alignes avec A* », il a adjoint l'equation $z\cdot g - 1 = 0$ avec $g = x_B^2 + y_B^2$ — rendre $g$ **inversible** — et la conclusion s'est reduite a zero dans l'ideal elargi. Le notebook 03, de son cote, verifie ses preuves de Wu par un test voisin : $\langle H,\ z J - 1,\ w g - 1\rangle = \langle 1\rangle$, l'« astuce de Rabinowitsch ».

Ces deux gestes manipulent le **meme objet sans jamais le nommer** : la **saturation** $I : g^\infty$ — l'ideal des conclusions valables sur le lieu non degenere $V(I)\setminus V(g)$. Ce carnet le nomme, le **calcule comme un ideal du plan** (pas seulement comme un test d'appartenance), derive le **Nullstellensatz fort** depuis le faible par la meme astuce, et ecrit enfin le contrat exact que la serie signe quand elle dit « demontre modulo les degenereescences ».

Fil rouge inchange : le theoreme du milieu de l'hypotenuse, dans son encodage a quatre points du 02 §6.


## Plan

1. §1 — Le fil rouge, troisieme visite : objets, rappel du geste du 02 §7.
2. §2 — Colon et saturation $I : g^n$, $I : g^\infty$ : la chaine calculee, puis l'ideal sature **exhibe** par elimination.
3. §3 — L'astuce de Ritt–Rabinowitsch relue : $\langle I, zg-1\rangle \cap k[x] = I : g^\infty$ — l'astuce EST l'algorithme.
4. §4 — Nullstellensatz : du faible au fort, la derivation classique **executee** avec cofacteurs sympy.
5. §5 — Le temoin negatif qui **survit** a la saturation (rectangle non isocele) — l'organe Z3 trouve un contre-exemple non degenere.
6. §6 — Le prix : trois recettes, deux moteurs, mesures.
7. §7 — Le contrat de la serie relu : chaque critere du 02, du 03 (Wu, Rabinowitsch) est un enonce de saturation.
8. §8–§10 — Trois exercices.


In [1]:
import time
import sympy as sp

t_start = time.perf_counter()
print("sympy", sp.__version__)


sympy 1.14.0


### §1 — Le fil rouge, troisieme visite

On reconstruit l'encodage du 02 §6 : $A$ a l'origine, les points $B, C, D$ libres, deux hypotheses d'orthogonalite, une conclusion d'alignement. Le theoreme est **faux** tel quel (temoin $B = A$ du 02 §6, ou les deux hypotheses sont trivialement nulles) et le 02 §7 l'a sauve en inversant $g = x_B^2 + y_B^2$ dans un anneau elargi.


In [2]:
# Les objets du fil rouge, comme au 02 §6
xB, yB, xC, yC, xD, yD = sp.symbols('x_B y_B x_C y_C x_D y_D')
PLAN = (xB, yB, xC, yC, xD, yD)          # les six inconnues du plan

I_gens = [xB * xC + yB * yC,             # H2a : AB scalaire AC = 0
          xB * xD + yB * yD]             # H2b : AB scalaire AD = 0
CONCL = xC * yD - xD * yC                # A, C, D alignes (produit vectoriel)
g_nd = xB ** 2 + yB ** 2                 # non-degenerescence B != A

print("I  = <", I_gens, ">")
print("CONCL =", CONCL, "| g_nd =", g_nd)


I  = < [x_B*x_C + y_B*y_C, x_B*x_D + y_B*y_D] >
CONCL = x_C*y_D - x_D*y_C | g_nd = x_B**2 + y_B**2


In [3]:
# Rappel du geste du 02 §7 : inverser g dans un anneau elargi
z = sp.symbols('z')
G_elargi = sp.groebner(I_gens + [z * g_nd - 1], *PLAN, z, order='lex')
reste_02 = G_elargi.reduce(CONCL)[1]
print("reste de CONCL dans <I, z*g-1> :", reste_02, "| nul ?", reste_02 == 0)
print("(c'est le verdict du 02 §7 : le theoreme tient sous B != A)")


reste de CONCL dans <I, z*g-1> : 0 | nul ? True
(c'est le verdict du 02 §7 : le theoreme tient sous B != A)


#### Lecture du resultat

Ce reste nul dit une chose precise mais detournée : $\mathrm{CONCL} \in \langle I,\ zg-1\rangle$, un ideal de l'anneau **a sept variables** $k[x_B,\dots,y_D, z]$. La variable $z$ n'a aucun sens geometrique — elle est la qui inverser $g$. Trois questions restent sans reponse :

- quel **ideal du plan** (six variables, sans $z$) ce reste nul atteste-t-il ?
- qu'est-ce que cette appartenance **garantit geometriquement** (sur quels points, dans quel corps) ?
- quel est le lien exact avec le test $\langle H, zJ-1, wg-1\rangle = \langle 1\rangle$ du 03 §7 ?

Les trois reponses portent le meme nom : la saturation.


### §2 — Le colon et la saturation : $I : g^\infty$

**Definition.** Pour un ideal $I$ et un polynome $g$ :

$$I : g^n = \{\, f \ :\ f\, g^n \in I \,\}, \qquad I : g^\infty = \bigcup_{n \geq 1} I : g^n.$$

La chaine $I : g \subseteq I : g^2 \subseteq \cdots$ croit et **stabilise** (l'anneau est noetherien) : l'union est un ideal. Lecture geometrique : $I : g^\infty$ est le plus grand ideal qui **coincide avec $I$ partout ou $g \neq 0$** — les conclusions valables sur le lieu non degenere $V(I) \setminus V(g)$.

Premiere visite : a la main, par la chaine. On cherche le plus petit $n$ tel que $\mathrm{CONCL}\cdot g^n \in I$ — c'est exactement ce que la pseudo-division du 03 etablit pour ses propres systemes.


In [4]:
# La chaine a la main : plus petit n tel que CONCL * g^n appartienne a I
G6 = sp.groebner(I_gens, *PLAN, order='lex')
for n in range(1, 4):
    restes = G6.reduce(sp.expand(CONCL * g_nd ** n))
    print(f"n = {n} : reste de CONCL*g^{n} par I : {restes[1]}  | nul ? {restes[1] == 0}")


n = 1 : reste de CONCL*g^1 par I : 0  | nul ? True
n = 2 : reste de CONCL*g^2 par I : 0  | nul ? True
n = 3 : reste de CONCL*g^3 par I : 0  | nul ? True


#### Lecture du resultat

Le reste s'annule des $n = 1$ : $\mathrm{CONCL}\cdot g \in I$, donc $\mathrm{CONCL} \in I : g \subseteq I : g^\infty$. Ce n'est pas un miracle — les cofacteurs sont explicites, et la machine peut les exhiber :


In [5]:
# Les cofacteurs explicites de l'appartenance CONCL*g in I
# deux identites de base, chacune verifiee par developpement
id1 = sp.expand(xB * CONCL - (yD * I_gens[0] - yC * I_gens[1]))
id2 = sp.expand(yB * CONCL - (xC * I_gens[1] - xD * I_gens[0]))
print("xB*CONCL = yD*H2a - yC*H2b  -> ecart :", id1, "| nul ?", id1 == 0)
print("yB*CONCL = xC*H2b - xD*H2a  -> ecart :", id2, "| nul ?", id2 == 0)
# en les combinant : g*CONCL = xB*(xB*CONCL) + yB*(yB*CONCL)
q1, q2 = xB * yD - yB * xD, yB * xC - xB * yC
ecart = sp.expand(CONCL * g_nd - (q1 * I_gens[0] + q2 * I_gens[1]))
print("cofacteurs : q1 =", q1, "| q2 =", q2)
print("CONCL*g - (q1*H2a + q2*H2b) =", ecart, "| nul ?", ecart == 0)


xB*CONCL = yD*H2a - yC*H2b  -> ecart : 0 | nul ? True
yB*CONCL = xC*H2b - xD*H2a  -> ecart : 0 | nul ? True
cofacteurs : q1 = x_B*y_D - x_D*y_B | q2 = -x_B*y_C + x_C*y_B
CONCL*g - (q1*H2a + q2*H2b) = 0 | nul ? True


#### Lecture du resultat

La chaine a repondu **pour cette conclusion la**. Mais un ideal est plus qu'un test d'appartenance : on veut les **generateurs** de $I : g^\infty$, un objet du plan a six variables, sans variable auxiliaire. C'est le travail de l'elimination :

$$I : g^\infty \;=\; \langle I,\ 1 - t\cdot g \rangle \ \cap\ k[x_B, \dots, y_D]$$

avec $t$ une variable fraiche, eliminee par une base de Groebner lexicographique (theoreme d'elimination : les elements de la base qui ne contiennent pas $t$ engendrent l'intersection).


In [6]:
# L'ideal sature exhibe par elimination : <I, 1 - t*g> intersecte k[PLAN]
t = sp.symbols('t')
G_sat = sp.groebner(I_gens + [t * g_nd - 1], *PLAN, t, order='lex')
sat_gens = [p.as_expr() if hasattr(p, 'as_expr') else p for p in G_sat.polys
            if not (p.as_expr().has(t) if hasattr(p, 'as_expr') else p.has(t))]
sat_gens = [sp.expand(p) for p in sat_gens]
print("generateurs de I : g^infini (sans t) :")
for p in sat_gens:
    print("   ", p)


generateurs de I : g^infini (sans t) :
    x_B*x_C + y_B*y_C
    x_B*x_D + y_B*y_D
    x_C*y_D - x_D*y_C


In [7]:
# Le theoreme, redevenu un enonce du plan : CONCL appartient-il a l'ideal sature ?
G_plane = sp.groebner(sat_gens, *PLAN, order='lex')
reste_sat = G_plane.reduce(CONCL)[1]
print("reste de CONCL par I : g^infini :", reste_sat, "| nul ?", reste_sat == 0)
print("generation du plan :", len(sat_gens), "polynomes, zero variable auxiliaire")


reste de CONCL par I : g^infini : 0 | nul ? True
generation du plan : 3 polynomes, zero variable auxiliaire


#### Lecture du resultat

Voila l'objet que les deux carnets precedents manipulaient sans le montrer : un ideal **du plan**, engendre par les hypotheses et (sous une forme ou une autre) la conclusion elle-meme. Le theoreme s'enonce maintenant sans variable artificielle :

$$\mathrm{CONCL} \in I : g_{nd}^\infty \quad\text{— « l'alignement est valable partout ou } B \neq A \text{ et les hypotheses tiennent »}.$$

Verification de coherence : la chaine et l'elimination doivent raconter la meme histoire. On calcule $I : g^n$ pour $n = 1, 2, 3$ (meme recette d'elimination, avec $g^n$) et on regarde la chaine se stabiliser **sur** l'ideal sature.


In [8]:
# Cross-validation : la chaine I:g^n stabilise sur l'ideal sature
def saturation_par_elimination(gens_ideal, g, variables, puissance=1):
    """Rend la liste des generateurs de <I, 1 - t*g^p> intersecte k[variables]."""
    u = sp.symbols('_u')
    G = sp.groebner(gens_ideal + [u * g ** puissance - 1], *variables, u, order='lex')
    return [sp.expand(p.as_expr()) for p in G.polys if not p.as_expr().has(u)]

chain = {}
for n in (1, 2, 3):
    chain[n] = saturation_par_elimination(I_gens, g_nd, PLAN, puissance=n)
    gn = sp.groebner(chain[n], *PLAN, order='lex')
    contient_concl = gn.reduce(CONCL)[1] == 0
    print(f"I : g^{n} : {len(chain[n])} generateurs | contient CONCL : {contient_concl}")

# stabilisation : chaque etage doit etre contenu dans le suivant, et le dernier = sature
for n in (1, 2):
    Gn, Gnext = sp.groebner(chain[n], *PLAN, order='lex'), sp.groebner(chain[n + 1], *PLAN, order='lex')
    incl = all(Gnext.reduce(p)[1] == 0 for p in chain[n])
    print(f"I : g^{n}  inclus dans I : g^{n+1} : {incl}")
G3_vs_sat = sp.groebner(chain[3], *PLAN, order='lex')
meme = (all(G3_vs_sat.reduce(p)[1] == 0 for p in sat_gens)
        and all(G_plane.reduce(p)[1] == 0 for p in chain[3]))
print("I : g^3 = I : g^infini (mutuellement) :", meme)


I : g^1 : 3 generateurs | contient CONCL : True
I : g^2 : 3 generateurs | contient CONCL : True
I : g^3 : 3 generateurs | contient CONCL : True
I : g^1  inclus dans I : g^2 : True
I : g^2  inclus dans I : g^3 : True
I : g^3 = I : g^infini (mutuellement) : True


#### Lecture du resultat

Trois facons de dire la meme chose, maintenant toutes **executees** : $\mathrm{CONCL}\cdot g \in I$ (la chaine), $\mathrm{CONCL} \in I:g^\infty$ (l'appartenance), et les generateurs de l'ideal sature (l'objet). Il reste a comprendre pourquoi l'astuce du 02 — ajouter $zg - 1$ — calcule *exactement* cet ideal.


### §3 — L'astuce de Ritt–Rabinowitsch, relue comme un algorithme

**Theoreme (Ritt–Rabinowitsch).** $\quad \langle I,\ zg - 1\rangle \cap k[x] \;=\; I : g^\infty$.

*Esquisse.* L'inclusion $\supseteq$ : si $f g^n = \sum a_i h_i$, multiplier par $z^n$ et retrancher $z^n g^n - 1 = (zg-1)(\dots)$ fait de $f$ une combination de $I$ et de $zg-1$, sans $z$ dans le resultat. L'inclusion $\subseteq$ : une identite $f = \sum a_i h_i + b(zg - 1)$, substituee en $z = 1/g$ puis multipliee par $g^m$ ($m$ assez grand), donne $f g^m \in I$. La verification machine tient dans deux reduce :


In [9]:
# Verification du theoreme sur le fil rouge, dans les deux sens
# sens 1 : les generateurs de l'ideal sature vivent dans <I, z*g-1> intersecte k[PLAN]
dans_elargi = all(G_elargi.reduce(p)[1] == 0 for p in sat_gens)
print("sat(I,g) inclus dans <I, z*g-1> cap k[PLAN] :", dans_elargi)

# sens 2 : CONCL*g (membre de I, donc de l'intersection) redonne le verdict du 02
identite_02 = G_elargi.reduce(CONCL)[1] == 0
print("CONCL dans <I, z*g-1> (rappel 02 §7) :", identite_02,
      "| CONCL dans I:g^infini (02b §2) :", reste_sat == 0)
print("deux routes, un seul verdict :", identite_02 == (reste_sat == 0))


sat(I,g) inclus dans <I, z*g-1> cap k[PLAN] : True
CONCL dans <I, z*g-1> (rappel 02 §7) : True | CONCL dans I:g^infini (02b §2) : True
deux routes, un seul verdict : True


#### Lecture du resultat — les trois criteres de la serie, un seul objet

| Carnet | Critere pose | Enonce de saturation equivalent | Force |
|---|---|---|---|
| 02 §7 | reste nul dans $\langle I, zg-1\rangle$ | $f \in I : g^\infty$ | forte |
| 03 §5 (Wu) | pseudo-reste nul dans la chaine | $f\, J^k \in I$, donc $f \in I : J^\infty$ | forte |
| 03 §7 (Rabinowitsch) | $\langle H, zJ{-}1, wg{-}1\rangle = \langle 1\rangle$ | $g \in \sqrt{I : J^\infty}$ | radiciel |

Les deux premiers critere prouvent une **appartenance a l'ideal sature**, le troisieme une appartenance a son **radical** — strictement plus faible en general ($f \in I:g^\infty \Rightarrow f^m \in I:g^\infty$... non : $f \in \sqrt{I:g^\infty}$ ne donne que $f^m \in I:g^\infty$). Le Nullstellensatz, section suivante, dit ce que le critere radiciel garantit quand meme — et pourquoi il suffit pour la question geometrique.


### §4 — Nullstellensatz : du faible au fort, la derivation executee

**Faible (Hilbert).** Sur $\mathbb{C}$ : un ideal propre $J \subsetneq k[x]$ a un zero commun : $V(J) \neq \varnothing$.

Le faible repond « le systeme a-t-il une solution ? » mais pas « quel polynome s'annule sur $V(J)$ ? ». La reponse est le **fort** :

$$I(V(J)) = \sqrt{J} \quad\text{— les polynomes nuls sur } V(J) \text{ sont exactement } \sqrt{J}.$$

La derivation classique tient en trois lignes, et la machine peut la *jouer* sur un exemple. Prenons $J = \langle x^2\rangle$, $f = x$ : $f$ s'annule sur $V(J) = \{0\}$, donc le fort promet $x \in \sqrt{J}$, c'est-a-dire $x^2 \in J$. D'ou vient cette puissance ?


In [10]:
# Etape 1 : le faible dit que <x^2, 1 - t*x> = <1> (aucun zero commun)
x, y_var = sp.symbols('x y')
J_toy = [x ** 2]
f_toy = x
G_ref = sp.groebner(J_toy + [1 - t * f_toy], x, t, order='lex')
print("base de <x^2, 1 - t*x> :", G_ref.polys if G_ref.polys else G_ref)
est_un = (len(G_ref.polys) == 1 and sp.expand(G_ref.polys[0].as_expr()) == 1)
print("l'ideal est <1> (faible NSZ) :", est_un)


base de <x^2, 1 - t*x> : [Poly(1, x, t, domain='ZZ')]
l'ideal est <1> (faible NSZ) : True


In [11]:
# Etape 2 : l'identite de Rabinowitsch, construite puis verifiee
b_cof = 1 + t * x              # degre 1 en t
a_cof = t ** 2                 # degre 2 en t
identite = sp.expand(sp.Integer(1) - (b_cof * (1 - t * x) + a_cof * x ** 2))
print("cofacteurs : b =", b_cof, "| a =", a_cof)
print("verification 1 - (b*(1-t*x) + a*x^2) =", identite, "| nul ?", identite == 0)

# la substitution t = 1/x, puis le nettoyage par x^m assez grand
m = max(sp.degree(b_cof, t), sp.degree(a_cof, t))
coeff_b = sp.expand(b_cof.subs(t, 1 / x) * x ** m)
coeff_a = sp.expand(a_cof.subs(t, 1 / x) * x ** m)
membre_droit = sp.expand(coeff_b * sp.Integer(0) + coeff_a * x ** 2)
print(f"apres substitution et nettoyage par x^{m} :")
print("   cofacteur polynomial devant x^2 :", coeff_a)
print("   x^2 =", membre_droit, "| certificat exact :",
      sp.expand(membre_droit - x ** 2) == 0)


cofacteurs : b = t*x + 1 | a = t**2
verification 1 - (b*(1-t*x) + a*x^2) = 0 | nul ? True
apres substitution et nettoyage par x^2 :
   cofacteur polynomial devant x^2 : 1
   x^2 = x**2 | certificat exact : True


In [12]:
# Etape 3 : le certificat obtenu, et la puissance minimale
G_J = sp.groebner(J_toy, x, order='lex')
p_min = next(p for p in range(1, 5) if G_J.reduce(f_toy ** p)[1] == 0)
print(f"la derivation livre f^{m} in J :", G_J.reduce(f_toy ** m)[1] == 0)
print(f"exposant minimal : {p_min} (x^{p_min - 1} in J :",
      G_J.reduce(f_toy ** (p_min - 1))[1] == 0, "— la puissance est necessaire)")


la derivation livre f^2 in J : True
exposant minimal : 2 (x^1 in J : False — la puissance est necessaire)


#### Lecture du resultat

Trois lignes de mathematiques, trois cellules de machine : le **faible** donne $1 \in \langle J, 1 - tf\rangle$ ; la **substitution** $t \mapsto 1/f$ suivie du nettoyage par $f^m$ transforme cette identite en certificat $f^m \in J$ ; c'est le **fort**. Aucune etape ne depend de l'exemple — l'astuce est la meme que celle du §3, appliquee au radical.

Pour la serie, le fort a une consequence de contrat : sur $\mathbb{C}$,

$$f \in \sqrt{I : g^\infty} \iff f \text{ s'annule sur } V(I) \setminus V(g).$$

Le critere radiciel du 03 §7 teste donc *exactement* la question geometrique « la conclusion s'annule-t-elle sur les figures non degenerees ? » — et le critere fort du 02 (l'appartenance sature) est **plus fort que necessaire** pour cette question. Verifions le contrat sur le fil rouge :


In [13]:
# Le contrat execute : CONCL s'annule sur V(I) prive de V(g) ?
w = sp.symbols('w')
G_contrat = sp.groebner(sat_gens + [1 - w * CONCL], *PLAN, w, order='lex')
est_un = (len(G_contrat.polys) == 1 and sp.expand(G_contrat.polys[0].as_expr()) == 1)
print("<I:g^infini, 1 - w*CONCL> = <1> :", est_un)
print("(vrai : aucun point non degenere ou les hypotheses tiennent et CONCL != 0)")


<I:g^infini, 1 - w*CONCL> = <1> : True
(vrai : aucun point non degenere ou les hypotheses tiennent et CONCL != 0)


### §5 — Le temoin negatif qui survit a la saturation

La saturation n'est pas une machine a prouver : elle **retire l'echappatoire degeneree**, rien d'autre. Si un enonce faux possede un contre-exemple **non degenere**, aucune saturation ne le sauvera. Retour au temoin du 02 §5 : « *un triangle rectangle est isocele en A* » — faux, temoin 3-4-5 choisi a la main. Soumettons-le a toute la machinerie, puis demandons a l'organe reel ([`real_geometry_bridge`](real_geometry_bridge.py), Z3) de trouver lui-meme un temoin et de dire s'il est degenere.


In [14]:
# Le faux theoreme du 02 §5, passe a la moulinette de la saturation
xA, yA, xB2, yB2, xC2, yC2 = sp.symbols('x_A y_A x_B2 y_B2 x_C2 y_C2')
PLAN3 = (xA, yA, xB2, yB2, xC2, yC2)
H_rect = (xB2 - xA) * (xC2 - xA) + (yB2 - yA) * (yC2 - yA)   # angle droit en A
Ciso = sp.expand((xB2 - xA) ** 2 + (yB2 - yA) ** 2
                 - ((xC2 - xA) ** 2 + (yC2 - yA) ** 2))       # AB = AC ?
g_rect = (xB2 - xA) ** 2 + (yB2 - yA) ** 2                    # B != A

sat_rect = saturation_par_elimination([H_rect], g_rect, PLAN3)
G_rect = sp.groebner(sat_rect, *PLAN3, order='lex')
reste_rect = G_rect.reduce(Ciso)[1]
print("reste de Ciso par sat(<H>, g) :", reste_rect, "| nul ?", reste_rect == 0)

w2 = sp.symbols('w2')
G_contre = sp.groebner(sat_rect + [1 - w2 * Ciso], *PLAN3, w2, order='lex')
est_un = (len(G_contre.polys) == 1 and sp.expand(G_contre.polys[0].as_expr()) == 1)
print("radicalement, Ciso s'annule-t-elle sur les figures non degenerees ?", est_un)


reste de Ciso par sat(<H>, g) : -2*x_A*x_B2 + 2*x_A*x_C2 + x_B2**2 - x_C2**2 - 2*y_A*y_B2 + 2*y_A*y_C2 + y_B2**2 - y_C2**2 | nul ? False
radicalement, Ciso s'annule-t-elle sur les figures non degenerees ? False


In [15]:
# L'organe reel : Z3, deux fois — sans puis avec la contrainte non degenere
import sys
from pathlib import Path
for _p in (Path.cwd(), Path.cwd() / "MyIA.AI.Notebooks" / "SymbolicAI" / "Lean" / "Geometry"):
    if (_p / "real_geometry_bridge.py").exists():
        sys.path.insert(0, str(_p))

import real_geometry_bridge as rbg
print("organe pret :", Path(rbg.__file__).name)

def decrire(verdict, etiquette):
    print(f"--- {etiquette}")
    print("   verdict :", verdict)
    if verdict.witness:
        print("   temoin :", {k.name: v for k, v in verdict.witness.items()})
        g_au = g_rect.subs(verdict.witness)
        print("   g au temoin (= |AB|^2) :", g_au, "| degenere (g = 0) ?", g_au == 0)

decrire(rbg.decide_real([H_rect], Ciso, list(PLAN3)),
        "sans contrainte : Z3 prend la porte degenere s'il peut")
g_CA = (xC2 - xA) ** 2 + (yC2 - yA) ** 2       # C != A
decrire(rbg.decide_real([H_rect], Ciso, list(PLAN3), assumptions=[g_rect, g_CA]),
        "avec les contraintes B != A et C != A")


organe pret : real_geometry_bridge.py
--- sans contrainte : Z3 prend la porte degenere s'il peut
   verdict : INVALID — temoin reel exact
   temoin : {'x_A': 0, 'y_A': 0, 'x_B2': 0, 'y_B2': 0, 'x_C2': 0, 'y_C2': -1}
   g au temoin (= |AB|^2) : 0 | degenere (g = 0) ? True
--- avec les contraintes B != A et C != A
   verdict : INVALID — temoin reel exact
   temoin : {'x_A': 0, 'y_A': 0, 'x_B2': -1, 'y_B2': 0, 'x_C2': 0, 'y_C2': -2}
   g au temoin (= |AB|^2) : 1 | degenere (g = 0) ? False


#### Lecture du resultat

Les deux moteurs racontent la meme histoire a trois voix. La saturation a fait son office : le reste n'est pas nul, le contrat radiciel echoue. L'organe reel, laisse libre, prend la porte degenere ($B = A$ : hypotheses trivialement nulles, conclusion violee) — exactement la porte que la saturation ferme. Mais contraint a $B \neq A$ **et** $C \neq A$, il trouve **quand meme** un temoin, avec $g \neq 0$ : un vrai triangle rectangle non isocele. La saturation avait raison de fermer la premiere porte ; elle ne peut rien contre la seconde.

C'est la ligne de partage honnete du cours : **saturer purge les degenerescences, elle ne soigne pas les enonces faux**.


### §6 — Le prix : trois recettes, deux moteurs

Trois recettes repondent a trois questions differentes, a des couts differents. Comme au 02 §8, on mesure :


In [16]:
# Mesures : appartenance (chaine), generateurs (elimination), contrat (radical)
t0 = time.perf_counter()
_ = sp.groebner(I_gens, *PLAN, order='lex').reduce(sp.expand(CONCL * g_nd))[1]
t1 = time.perf_counter()
saturation_par_elimination(I_gens, g_nd, PLAN)
t2 = time.perf_counter()
zp, wp = sp.symbols('_zp _wp')
_ = sp.groebner(I_gens + [zp * g_nd - 1, wp * CONCL - 1], *PLAN, zp, wp, order='lex')
t3 = time.perf_counter()

print("recette 1 : appartenance f*g^n in I   (6 var) : %.3f s" % (t1 - t0))
print("recette 2 : generateurs de I:g^infini (7 var) : %.3f s" % (t2 - t1))
print("recette 3 : contrat radiciel           (8 var) : %.3f s" % (t3 - t2))


recette 1 : appartenance f*g^n in I   (6 var) : 0.001 s
recette 2 : generateurs de I:g^infini (7 var) : 0.002 s
recette 3 : contrat radiciel           (8 var) : 0.003 s


#### Lecture du resultat

| Recette | Question a laquelle elle repond | Cout |
|---|---|---|
| chaine $f g^n \in I$ | « cette conclusion precise est-elle prouvee ? » | le moins cher — un test, pas un objet |
| elimination $\langle I, 1-tg\rangle \cap k[x]$ | « quel est l'ideal sature ? » | une variable de plus — mais on garde l'objet |
| radical $\langle 1\rangle$ | « la conclusion s'annule-t-elle geometriquement ? » | deux variables de plus — la question la plus faible, pas la plus chere a poser |

Le choix depend de la question ; aucune recette ne domine les autres. C'est la meme lecon que le 02 §8, un etage plus haut.


### §7 — Le contrat de la serie, relu d'un seul endroit

Toutes les methodes de la series disent maintenant la meme chose en des forces differentes :

| Position | Critere | Traduit en saturation | Portee |
|---|---|---|---|
| [02](Geometry-02-From-Equation-To-Proof.ipynb) §7 | reste nul dans $\langle I, zg-1\rangle$ | $f \in I : g^\infty$ | ideal sature, sur $\mathbb{C}$ |
| [03](Geometry-03-Wu-Method-Python.ipynb) §5 | pseudo-reste nul (chaine) | $f\,J^k \in I$, donc $f \in I:J^\infty$ | ideal sature, par morceaux (chaine) |
| [03](Geometry-03-Wu-Method-Python.ipynb) §7 | $\langle H, zJ{-}1, wg{-}1\rangle = \langle 1\rangle$ | $g \in \sqrt{I:J^\infty}$ | geometrique, sur $\mathbb{C}$ |
| [03b](Geometry-03b-Ritt-Decomposition-Python.ipynb) | scission de Ritt | decomposition du lieu, saturation par composante | les bords ou l'enonce est muet |
| [05](Geometry-05-Pont-Formel-Python.ipynb) | preuve Lean du lac | l'hypothese $B \neq A$ y est explicite dans l'enonce | noyau, toutes figures concretes |

Une seule cellule pour derouler le fil rouge devant cette table :


In [17]:
# Le fil rouge, trois verdicts d'un coup
v_chaine = sp.groebner(I_gens, *PLAN, order='lex').reduce(sp.expand(CONCL * g_nd))[1] == 0
v_ideal = sp.groebner(sat_gens, *PLAN, order='lex').reduce(CONCL)[1] == 0
G_rad = sp.groebner(I_gens + [zp * g_nd - 1, wp * CONCL - 1], *PLAN, zp, wp, order='lex')
v_radical = (len(G_rad.polys) == 1 and sp.expand(G_rad.polys[0].as_expr()) == 1)
print("chaine (Wu/02)     : CONCL*g in I          :", v_chaine)
print("ideal sature (02b) : CONCL in I:g^infini   :", v_ideal)
print("contrat (03 §7)    : vanishing non degenere :", v_radical)
print("coherence :", v_chaine and v_ideal and v_radical)


chaine (Wu/02)     : CONCL*g in I          : True
ideal sature (02b) : CONCL in I:g^infini   : True
contrat (03 §7)    : vanishing non degenere : True
coherence : True


### §8 — Exercice 1 : saturations partielles, ou le theoreme a plus d'une porte

Le 02 §11 laissait la question ouverte : que donnent les saturations **partielles** par $x_B$ seul, puis $y_B$ seul ? Montrer que chacune suffit a contenir la conclusion, puis expliquer pourquoi le 03 prefere saturer par le **produit** des non-degenerescences.


In [18]:
# Exercice 1 : CONCL appartient-il a I : x_B^infini ? a I : y_B^infini ?
# Etape 1 : calculer les deux ideaux satures partiels par elimination
# Etape 2 : reduire CONCL par chacun
# Etape 3 : pourquoi le produit des deux composantes (B != A) est-il le geste general ?
reste_xB = None  # TODO etudiant : reste de CONCL par sat(I, x_B)
reste_yB = None  # TODO etudiant : reste de CONCL par sat(I, y_B)
print("Exercice 1 : deux saturations partielles")


Exercice 1 : deux saturations partielles


In [19]:
# Verification de l'exercice 1
sat_xB = saturation_par_elimination(I_gens, xB, PLAN)
sat_yB = saturation_par_elimination(I_gens, yB, PLAN)
ref_xB = sp.groebner(sat_xB, *PLAN, order='lex').reduce(CONCL)[1]
ref_yB = sp.groebner(sat_yB, *PLAN, order='lex').reduce(CONCL)[1]
print("reference : reste de CONCL par sat(I, x_B) :", ref_xB, "| nul ?", ref_xB == 0)
print("reference : reste de CONCL par sat(I, y_B) :", ref_yB, "| nul ?", ref_yB == 0)
if reste_xB is not None:
    print("votre reste x_B :", reste_xB, "| conforme :", bool(reste_xB) == (ref_xB == 0))
if reste_yB is not None:
    print("votre reste y_B :", reste_yB, "| conforme :", bool(reste_yB) == (ref_yB == 0))


reference : reste de CONCL par sat(I, x_B) :

 0 | nul ? True
reference : reste de CONCL par sat(I, y_B) : 0 | nul ? True


#### Lecture de l'exercice 1

Chaque saturation partielle suffit : sur le lieu $x_B \neq 0$, les hypotheses forcent $C$ et $D$ sur une meme droite issue de $A$ (et symetriquement sur $y_B \neq 0$). Les deux lieux couvrent $V(I) \setminus \{B = A\}$ — le theoreme tient sur l'union. Mais un seul enonce par **produit** $x_B \cdot y_B \neq 0$ serait faux (il exclurait trop) : c'est $g_{nd} = x_B^2 + y_B^2$ qui partitionne exactement, et c'est ce genre de partition que le produit des initiales de Wu realise systematiquement (03 §6).


### §9 — Exercice 2 : un ideal que la saturation transforme vraiment

Sur le fil rouge, l'ideal sature ne differe de $I$ que par la conclusion. Ce n'est pas toujours aussi discret. Prenons $I = \langle x y,\ x z\rangle$ et $g = y$ :

1. calculer $I : y^\infty$ par elimination ;
2. verifier que l'inclusion $I \subsetneq I : y^\infty$ est **stricte** ;
3. lire geometriquement : $V(I)$ est l'union de deux lieux — lequel survive-t-il apres avoir retire $V(y)$ ?


In [20]:
# Exercice 2 : saturation stricte
# Etape 1 : I:y^infini par elimination (3 variables x, y, z puis variable auxiliaire)
# Etape 2 : trouver un polynome dans I:y^infini qui n'est pas dans I
# Etape 3 : decrire V(I) et ce que la saturation par y en retire
sat_ex2 = None   # TODO etudiant : generateurs de <x*y, x*z> : y^infini
temoin_ex2 = None  # TODO etudiant : un polynome de I:y^infini hors de I
print("Exercice 2 : la saturation stricte")


Exercice 2 : la saturation stricte


In [21]:
# Verification de l'exercice 2
xa, ya, za = sp.symbols('x_a y_a z_a')
I_ex2 = [xa * ya, xa * za]
sat_ref = saturation_par_elimination(I_ex2, ya, (xa, ya, za))
print("reference : I : y^infini =", sat_ref)
G_I2 = sp.groebner(I_ex2, xa, ya, za, order='lex')
hors = [p for p in sat_ref if G_I2.reduce(p)[1] != 0]
print("generateurs satures hors de I :", hors)
if sat_ex2 is not None:
    G_e2 = sp.groebner(list(sat_ex2), xa, ya, za, order='lex')
    ok = all(G_e2.reduce(p)[1] == 0 for p in sat_ref) and all(
        sp.groebner(sat_ref, xa, ya, za, order='lex').reduce(p)[1] == 0 for p in sat_ex2)
    print("votre ideal coincide avec la reference :", ok)
if temoin_ex2 is not None:
    print("votre temoin est hors de I :", G_I2.reduce(sp.expand(temoin_ex2))[1] != 0)


reference : I : y^infini = [x_a]
generateurs satures hors de I : [x_a]


#### Lecture de l'exercice 2

$V(x y, x z)$ est l'union du plan $\{x = 0\}$ et de la droite $\{y = z = 0\}$. Retirer $V(y)$ (le plan $y = 0$) coupe la droite en un point mais ne touche pas au plan $\{x=0\}$ hors de $y=0$ ; la cloture du reste est le plan entier $\{x = 0\}$ — et $I : y^\infty = \langle x\rangle$. La saturation a **supprime une composante** de l'ideal : sur le fil rouge elle ajoutait la conclusion, ici elle retire de l'hypothese ce qui ne vivait que sur la partie degeneree.


### §10 — Exercice 3 : votre premier certificat de Nullstellensatz

Avec $J = \langle x^2,\ x y\rangle$ et $f = x$ : $f$ s'annule sur $V(J)$.

1. trouver le plus petit $m$ tel que $f^m \in J$ ;
2. construire une identite $1 = b\,(1 - t x) + a\,x^2$ (celle du §4 fonctionne telle quelle), la verifier par developpement, puis substituer $t = 1/x$ et nettoyer par $x^m$ — l'identite finale doit etre un certificat de $x^m \in J$.


In [22]:
# Exercice 3 : le certificat complet
# Etape 1 : minimalite de m (tester m = 1, 2, 3 par reduce contre GB(J))
# Etape 2 : reduce(1) sur <J, 1 - t*x>, substitution t = 1/x, multiplication par x^m
# Etape 3 : verifier que la somme obtenue redonne x^m comme combination de x^2 et x*y
m_ex3 = None       # TODO etudiant : plus petit m tel que x^m in <x^2, x*y>
cofacteurs_ex3 = None  # TODO etudiant : les cofacteurs (devant x^2 et x*y) de l'identite finale
print("Exercice 3 : certificat de Nullstellensatz")


Exercice 3 : certificat de Nullstellensatz


In [23]:
# Verification de l'exercice 3
xb_, yb_ = sp.symbols('x_b y_b')
J_ex3 = [xb_ ** 2, xb_ * yb_]
G_ex3 = sp.groebner(J_ex3, xb_, yb_, order='lex')
p_min = next(p for p in (1, 2, 3) if G_ex3.reduce(xb_ ** p)[1] == 0)
print(f"reference : exposant minimal m = {p_min} | x^{p_min - 1} in J :",
      G_ex3.reduce(xb_ ** (p_min - 1))[1] == 0)

# derivation : 1 = (1+t*x)*(1-t*x) + t^2*x^2, substitution t = 1/x, nettoyage
m_ref = max(sp.degree(1 + t * xb_, t), sp.degree(t ** 2, t))
coeff_a = sp.expand((t ** 2).subs(t, 1 / xb_) * xb_ ** m_ref)
cert = sp.expand(coeff_a * xb_ ** 2)
print(f"reference : derivation -> x^{m_ref} =", cert,
      "| certificat :", sp.expand(cert - xb_ ** m_ref) == 0)
if m_ex3 is not None:
    print("votre m :", m_ex3, "| conforme :", m_ex3 == p_min)
if cofacteurs_ex3 is not None:
    exp = xb_ ** m_ex3 if m_ex3 is not None else xb_ ** m_ref
    ident = sp.expand(sum(c * g for c, g in zip(cofacteurs_ex3, J_ex3)))
    print("votre identite somme c_i*g_i =", ident, "| exacte :", sp.expand(ident - exp) == 0)


reference : exposant minimal m = 2 | x^1 in J : False
reference : derivation -> x^2 = x_b**2 | certificat : True


#### Lecture de l'exercice 3

$x$ n'est pas dans $J$ (reste non nul), $x^2$ y est — $m = 2$. Le certificat final n'est pas un artefact : chaque coefficient de l'identite est polynomial, et la somme reconstruite redonne exactement $x^2$ comme combination de $x^2$ et $xy$. Vous venez d'executer, sur un exemple que vous choisissez, la preuve du Nullstellensatz fort — la meme que celle qui garantit les contrats radiciels du 03.


## Conclusion

Ce carnet a nomme et calcule l'objet que la serie manipulait depuis le 02 :

| Brique | Role | Ou |
|---|---|---|
| chaine $I : g^n$ | le test d'appartenance — ce que Wu etablit | §2 |
| elimination $\langle I, 1-tg\rangle \cap k[x]$ | les **generateurs** de l'ideal sature | §2 |
| theoreme de Ritt–Rabinowitsch | l'astuce du 02/03 EST l'algorithme de saturation | §3 |
| derivation faible $\to$ fort | Nullstellensatz execute, cofacteurs a l'appui | §4 |
| contrat radiciel | ce que « prouve modulo degenereescences » garantit sur $\mathbb{C}$ | §4, §7 |
| temoin non degenere | la saturation ne soigne pas les enonces faux — l'organe Z3 trouve le temoin | §5 |

**Limites honnetes.** (1) Le contrat radiciel vit sur $\mathbb{C}$ ; la subtilite reel vs complexe reste celle du 02 §12 et de l'organe `real_geometry_bridge`. (2) Quand la conclusion est un **quotient** et non un polynome, la scission de Ritt du 03b decompose ce que la saturation globalise — les deux lectures se completent. (3) Le lac Lean du 05 prouve l'enonce avec l'hypothese $B \neq A$ ** explicite** : la preuve au noyau ne sature rien, elle hypotheque — c'est une force, pas un doublon.


## Cout et reproductibilite

- CPU uniquement (sympy, Z3 via l'organe de la serie) ; aucune API, aucun GPU, aucun reseau ; deterministe (aucun tirage aleatoire).
- Detail chiffre dans le bloc `metadata.cost` du carnet.


## References

- J. L. Rabinowitsch, *Zum Hilbertschen Nullstellensatz* (1929) — l'astuce d'une variable, dans sa lettre originale.
- D. Cox, J. Little, D. O'Shea, *Ideals, Varieties, and Algorithms* (4e ed.), chap. 2 §2 (Nullstellensatz) et chap. 4 §4 (saturation, elimination) — la derivation executee en §4 en est le Theoreme 2.2.10 pousse en machine.
- D. Eisenbud, *Commutative Algebra with a View Toward Algebraic Geometry*, chap. 4 — saturation et decomposition primaire.
- S.-C. Chou, *Mechanical Geometry Theorem Proving* (1988) — la lecture « non-degenerescences » de la methode de Wu.

Serie : [Geometry](README.md) — la preuve automatique en geometrie, de la figure a la preuve. Suivant logique : [03b](Geometry-03b-Ritt-Decomposition-Python.ipynb) pour la decomposition, [05](Geometry-05-Pont-Formel-Python.ipynb) pour le pont au noyau.
